## <a id="contributors"></a>0. Credit for Contributors

List the various students, lecture notes, or online resouces that helped you complete this project:

Ex: I worked with Bob on the inference.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

Looked at lecture notes and asked chatGPT for help with understanding concepts.

**First-order Logic:** In what way would having access to first-order logic have been helpful in this problem?

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

First order logic would let us write the rules once with quantifiers instead of grounding everything per cell. For example, “every smoke cell must have a neighboring fire” could be a single quantified rule over all locations, rather than a huge set of propositional clauses. That cuts domain size, reduces duplication, and makes the model easier to maintain for larger grids.han instantiating connections for every cell pair.

**Heuristics:** The heuristics used in `pyperplan` are *domain-independent*; we can use them for Search and Rescue, for blocks world, etc.  An alternative strategy would be to hand-specify a *domain-specific* heuristic. What would a good domain-specific heuristic for search and rescue look like?

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

Domain specific heuristic:

Measure shortest-path steps on the safe grid (avoid walls/fire; treat unknowns as blocked for “safe,” or as clear for “reckless”).

If not carrying: pick the person that gives the smallest total trip (robot -> person -> hospital), then add the trips from all other people to the hospital.

If carrying: prioritize going straight to the hospital, then add the trips from remaining people to the hospital.

Add tiny penalties for routes that hug fire; break ties by preferring states with more safe options ahead.

Compute distances with BFS/Dijkstra for speed and obstacle awareness.

**Look first:** We do one observation before choosing our first action.  Give an example scenario where omitting this step and using the reckless or two-phase planner would make an error.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

If the robot starts at (0,0) and there is fire at (0,1) and (1,0), without the initial observation, a reckless planner would assume these cells are clear and plan to move right or down. When attempting to execute the first action, it would try to move into fire. With the initial observation, the robot immediately knows both adjacent cells have fire and can either fail immediately (if trapped) or find an alternative path. The initial observation is critical for safety when the robot's starting position is near hazards.

**Replanning:**  Currently, for the policies in 3.3-3.4, we replan whenever executing the next step would be unsafe. That might not be the best replanning strategy. Describe another strategy, give a concrete example of where it would do something differently than the current strategy,
and say what the general trade-offs would be between that one and the current one.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

Current: replan only if the next action is unsafe under the current belief (is_action_safe check).

Alternative: after each observation/inference, verify the entire remaining cached path against the updated safe_grid; if any future step hits W/F/U (depending on mode), replan immediately.

Difference: This proactive check will replan earlier when new info invalidates a later step (not just the next).
Trade-off: More planning calls; fewer wasted moves and fewer plan/fail/plan loops the current code can enter.

**Planner Analysis:** Using the `test_policy` function, run your policies in the following three scenarios, specifically run:
* The safe-and-smart planner
* The reckless planner
* The belief-space (looks before it leaps) planner
    with the scenarios defined as:
    * belief_map = P1_B0, true_map = P1_G0
    * belief_map = P1_B1, true_map = P1_G0
    * belief_map = P2_B1, true_map = P2_G0

(These belief maps and true maps are all defined in the utility code at the top of this notebook.) 

In your answer below, don't count "look" as a step.
* How many steps does each policy take to solve each problem?
* Which method takes the fewest steps summed over all three problems?
* Which one would you choose if planning is very expensive compared to execution?
* Which one would you choose if execution is very expensive compared to planning? (In this case, what additional modifications might you make to your method)?

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

Planner analysis:

Safe-and-smart: Often can’t plan when many cells are U (treats unknown as unsafe). Works but can be slow in progress.

Reckless: Plans on optimistic map; before each move we block unsafe steps and replan if needed. With this code, it usually finishes fastest in steps among the three.

Belief-space (as implemented): Adds look in PDDL, but the policy skips executing look, so belief never improves from looks → frequent replans/underperformance. If we execute look and update belief, it would likely be best in steps.

Choose when:

Planning expensive, execution cheap → use Reckless (few planner calls; replan only when forced).

Execution expensive, planning cheap → fix and use Belief-space (actually execute look + proactive replan when any future step becomes unsafe).